In [53]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent          # notebook lives in notebooks/, so the root is one level up
RAW_PATH = ROOT / "data" / "raw" / "hotel_bookings.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
VISUALS_DIR = ROOT / "visuals"

In [54]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()
print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [56]:
miss = df.isna().sum()
miss[miss>0]

children         4
country        488
agent        16340
company     112593
dtype: int64

In [57]:
print(df.duplicated().sum())


31994


In [58]:
print(df.duplicated().sum())      # expect 31,994
df = df.drop_duplicates().reset_index(drop=True)

31994


In [59]:
df["children"]=df["children"].fillna(0).astype(int)
df["country"]=df["country"].fillna("Unknown")
df["has_agent"] =df["agent"].notna().astype(int)
df["has_company"] =df["company"].notna().astype(int)
df = df.drop(columns=["agent", "company"])

df["meal"] = df['meal'].replace("Undefined","SC")

In [60]:
no_guests = (df["adults"] + df["children"] + df["babies"]) == 0
print(no_guests.sum())            
df = df[~no_guests]


166


In [61]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,87386,87387,87388,87389,87390,87391,87392,87393,87394,87395
hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,Resort Hotel,...,City Hotel,City Hotel,City Hotel,City Hotel,City Hotel,City Hotel,City Hotel,City Hotel,City Hotel,City Hotel
is_canceled,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
lead_time,342,737,7,13,14,0,9,85,75,23,...,44,188,135,164,21,23,102,34,109,205
arrival_date_year,2015,2015,2015,2015,2015,2015,2015,2015,2015,2015,...,2017,2017,2017,2017,2017,2017,2017,2017,2017,2017
arrival_date_month,July,July,July,July,July,July,July,July,July,July,...,August,August,August,August,August,August,August,August,August,August
arrival_date_week_number,27,27,27,27,27,27,27,27,27,27,...,35,35,35,35,35,35,35,35,35,35
arrival_date_day_of_month,1,1,1,1,1,1,1,1,1,1,...,31,31,30,31,30,30,31,31,31,29
stays_in_weekend_nights,0,0,0,0,0,0,0,0,0,0,...,1,2,2,2,2,2,2,2,2,2
stays_in_week_nights,0,0,1,1,2,2,2,3,3,4,...,3,3,4,4,5,5,5,5,5,7
adults,2,2,1,1,2,2,2,2,2,2,...,2,2,3,2,2,2,3,2,2,2


In [62]:
zero_nights = (df["stays_in_weekend_nights"] + df["stays_in_week_nights"]) == 0
print(zero_nights.sum())

df = df[~zero_nights]


bad_price = (df["adr"] < 0 )| (df["adr"] > 1000)
print(bad_price.sum())
df = df[~bad_price].reset_index(drop=True)



591
2


In [63]:
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"]+df["babies"]
df["is_family"] = ((df["children"]+df["babies"])>0).astype(int)
df["revenue"]= df["adr"]*df["total_nights"]
df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)

In [64]:
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

df["arrival_month_num"] = df["arrival_date_month"].map(month_map)

In [65]:
df[["arrival_date_month", "arrival_month_num"]].head()
df["arrival_month_num"].isna().sum()     # must be 0

np.int64(0)

In [66]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

df["season"] = df["arrival_month_num"].map(season_map)

In [67]:
df["lead_time_group"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, 365, 10000],
    labels=["0-7d", "8-30d", "31-90d", "91-180d", "181-365d", "365d+"]
)

In [68]:
df["lead_time_group"].value_counts().sort_index()
df["lead_time_group"].isna().sum()       # must be 0

np.int64(0)

In [69]:
df["arrival_date"] = pd.to_datetime(dict(
    year=df["arrival_date_year"],
    month=df["arrival_month_num"],
    day=df["arrival_date_day_of_month"]
))
df["arrival_weekday"] = df["arrival_date"].dt.day_name()
print(df.shape)       # now (86637, 42)

(86637, 42)


In [70]:
print(df.isna().sum().sum())      # expect 0
print(df.shape)                   # expect (86637, 42)
print(df["is_canceled"].mean())   # expect about 0.277

0
(86637, 42)
0.2768447660930088


In [71]:
df.to_csv(PROCESSED_DIR / "hotel_bookings_clean.csv", index=False)